In [1]:
import argparse
import os
import jax
import torch 
import numpy as np
import pickle
from quantum_transformers.datasets import get_custom_classification_dataloader
from quantum_transformers.transformers import Transformer
from quantum_transformers.quantum_layer_modified import (
    get_circuit, 
    basic_vqc, 
    angle_embedding,
    phase_embedding,
    amplitude_embedding,
    iqp_embedding
)
from quantum_transformers.training import train_and_evaluate
from typing import Callable, Tuple
import tensorcircuit as tc
import jax.numpy as jnp
import flax.linen as nn

2026-06-08 14:26:36.119881: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-08 14:26:36.119910: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-08 14:26:36.119926: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-08 14:26:36.981610: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Please first ``pip install -U qiskit`` to enable related functionality in translation module


In [2]:
print("JAX devices:", jax.devices())

JAX devices: [CudaDevice(id=0)]


In [3]:
HIDDEN_SIZE = 8      
NUM_HEADS = 2
NUM_BLOCKS = 2
MLP_HIDDEN = 8             
MAX_SEQ_LEN = 32
BATCH_SIZE = 16   ## try to increase again
NUM_EPOCHS = 40       
LEARNING_RATE = 1e-3
NUM_LAYERS_VQC = 2

DATA_PATHS = {
    'mc': {
        'type': 'mc_rp',
        'train': 'data/mc_train_data.txt',
        'val': 'data/mc_dev_data.txt',
        'test': 'data/mc_test_data.txt',
        'tokenizer': 'wordlevel',
        'vocab': 17,  
        'shared_files': None
    },
    'rp': {
        'type': 'mc_rp',
        'train': 'data/rp_train_data.txt',
        'val': None,
        'test': 'data/rp_test_data.txt',
        'tokenizer': 'wordlevel',
        'vocab': 115, 
        'shared_files': None
    }
}

SENTIMENT_FILES = [
    'data/imdb_labelled.txt', 
    'data/amazon_cells_labelled.txt', 
    'data/yelp_labelled.txt'
]

In [4]:
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


In [5]:
dataset_config = DATA_PATHS['mc']

train_loader, val_loader, test_loader, tokenizer = get_custom_classification_dataloader(
                dataset_type='mc_rp',
                train_path=dataset_config['train'],
                val_path=dataset_config['val'],
                test_path=dataset_config['test'],
                batch_size=BATCH_SIZE, # Use dynamic batch size
                max_seq_len=MAX_SEQ_LEN,
                tokenizer_type=dataset_config['tokenizer'], 
                vocab_size=dataset_config['vocab'],         
                tokenizer_files=None             
            ) 

Data Loaded. Train: 70, Val: 30, Test: 30
Tokenizer (wordlevel) trained. Vocab size: 19


In [6]:
circuit_fn = get_circuit(
    embedding=angle_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)



In [7]:
model_0 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

In [8]:
print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_0,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)

Starting Training...
Starting training for 40 epochs (Seed: 0)...


Epoch 1 | Train Loss: 0.6612 | Val Loss: 0.8070, Val AUC: 0.7177


Epoch 2 | Train Loss: 0.6931 | Val Loss: 0.7949, Val AUC: 0.7368


Epoch 3 | Train Loss: 0.6902 | Val Loss: 0.7722, Val AUC: 0.7560


Epoch 4 | Train Loss: 0.6715 | Val Loss: 0.7491, Val AUC: 0.7751


...

Epoch 37 | Train Loss: 0.6599 | Val Loss: 0.6955, Val AUC: 0.9952


Epoch 38 | Train Loss: 0.6432 | Val Loss: 0.6859, Val AUC: 0.9952


Epoch 39 | Train Loss: 0.6463 | Val Loss: 0.6779, Val AUC: 0.9952


Epoch 40 | Train Loss: 0.6382 | Val Loss: 0.6755, Val AUC: 1.0000
Total training time = 97.70s, Best AUC = 100.00% at epoch 40


Test Loss = 0.6565, Test AUC = 97.33%


In [9]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=angle_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───────┐┌───────┐                                                       »
q_0: ┤ Ry(0) ├┤ Rx(0) ├──■────────────────────────────────────────────────────»
     ├───────┤├───────┤┌─┴─┐     ┌───────┐                                    »
q_1: ┤ Ry(0) ├┤ Rx(0) ├┤ X ├──■──┤ Rx(0) ├────────────────────────────────────»
     ├───────┤├───────┤└───┘┌─┴─┐└───────┘┌───────┐                           »
q_2: ┤ Ry(0) ├┤ Rx(0) ├─────┤ X ├────■────┤ Rx(0) ├───────────────────────────»
     ├───────┤├───────┤     └───┘  ┌─┴─┐  └───────┘┌───────┐                  »
q_3: ┤ Ry(0) ├┤ Rx(0) ├────────────┤ X ├──────■────┤ Rx(0) ├──────────────────»
     ├───────┤├───────┤            └───┘    ┌─┴─┐  └───────┘┌───────┐         »
q_4: ┤ Ry(0) ├┤ Rx(0) ├─────────────────────┤ X ├──────■────┤ Rx(0) ├─────────»
     ├───────┤├───────┤                     └───┘    ┌─┴─┐  └───────┘┌───────┐»
q_5: ┤ Ry(0) ├┤ Rx(0) ├──────────────────────────────┤ X ├──────■────┤ Rx(0) ├»
     ├───────┤├───────┤                 

## Phase Embedding

In [10]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=phase_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_1 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_1,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...
Starting training for 40 epochs (Seed: 0)...


Epoch 1 | Train Loss: 0.7133 | Val Loss: 0.8248, Val AUC: 0.7560


Epoch 2 | Train Loss: 0.7080 | Val Loss: 0.7794, Val AUC: 0.7703


Epoch 3 | Train Loss: 0.6973 | Val Loss: 0.7488, Val AUC: 0.7751


Epoch 4 | Train Loss: 0.6695 | Val Loss: 0.7291, Val AUC: 0.7895


...

Epoch 37 | Train Loss: 0.6480 | Val Loss: 0.6850, Val AUC: 0.9952


Epoch 38 | Train Loss: 0.6423 | Val Loss: 0.6805, Val AUC: 0.9952


Epoch 39 | Train Loss: 0.6583 | Val Loss: 0.6827, Val AUC: 0.9952


Epoch 40 | Train Loss: 0.6413 | Val Loss: 0.6762, Val AUC: 0.9952
Total training time = 102.54s, Best AUC = 99.52% at epoch 36


Test Loss = 0.6647, Test AUC = 96.00%


In [11]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=phase_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───┐┌───────┐┌───┐┌───────┐                                              »
q_0: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├──■───────────────────────────────────────────»
     ├───┤├───────┤├───┤├───────┤┌─┴─┐     ┌───────┐                           »
q_1: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├┤ X ├──■──┤ Rx(0) ├───────────────────────────»
     ├───┤├───────┤├───┤├───────┤└───┘┌─┴─┐└───────┘┌───────┐                  »
q_2: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├─────┤ X ├────■────┤ Rx(0) ├──────────────────»
     ├───┤├───────┤├───┤├───────┤     └───┘  ┌─┴─┐  └───────┘┌───────┐         »
q_3: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├────────────┤ X ├──────■────┤ Rx(0) ├─────────»
     ├───┤├───────┤├───┤├───────┤            └───┘    ┌─┴─┐  └───────┘┌───────┐»
q_4: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├─────────────────────┤ X ├──────■────┤ Rx(0) ├»
     ├───┤├───────┤├───┤├───────┤                     └───┘    ┌─┴─┐  └───────┘»
q_5: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├──────────────────────────────┤ X ├──────■────»
     ├───┤├───────┤├───┤├───

# Amplitude Embedding

In [12]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=amplitude_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_2 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_2,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...


AttributeError: 'Circuit' object has no attribute 'initialize'

In [ ]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=amplitude_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

# IQP Embedding

In [13]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=iqp_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_3 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_3,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...
Starting training for 40 epochs (Seed: 0)...


Epoch 1 | Train Loss: 0.6758 | Val Loss: 0.7034, Val AUC: 0.7703


Epoch 2 | Train Loss: 0.6796 | Val Loss: 0.7091, Val AUC: 0.7751


Epoch 3 | Train Loss: 0.6787 | Val Loss: 0.7191, Val AUC: 0.7990


Epoch 4 | Train Loss: 0.6683 | Val Loss: 0.7198, Val AUC: 0.8182


...

Epoch 37 | Train Loss: 0.6195 | Val Loss: 0.6698, Val AUC: 0.9904


Epoch 38 | Train Loss: 0.6282 | Val Loss: 0.6708, Val AUC: 0.9904


Epoch 39 | Train Loss: 0.6286 | Val Loss: 0.6704, Val AUC: 0.9904


Epoch 40 | Train Loss: 0.6152 | Val Loss: 0.6699, Val AUC: 0.9952
Total training time = 238.34s, Best AUC = 99.52% at epoch 40


Test Loss = 0.6462, Test AUC = 99.56%


In [14]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=iqp_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───┐┌───────┐                                                    »
q_0: ┤ H ├┤ Rz(0) ├──■─────────────■────■─────────────■────■───────────»
     ├───┤├───────┤┌─┴─┐┌───────┐┌─┴─┐  │             │    │           »
q_1: ┤ H ├┤ Rz(0) ├┤ X ├┤ Rz(0) ├┤ X ├──┼─────────────┼────┼──────■────»
     ├───┤├───────┤└───┘└───────┘└───┘┌─┴─┐┌───────┐┌─┴─┐  │    ┌─┴─┐  »
q_2: ┤ H ├┤ Rz(0) ├───────────────────┤ X ├┤ Rz(0) ├┤ X ├──┼────┤ X ├──»
     ├───┤├───────┤                   └───┘└───────┘└───┘┌─┴─┐┌─┴───┴─┐»
q_3: ┤ H ├┤ Rz(0) ├──────────────────────────────────────┤ X ├┤ Rz(0) ├»
     ├───┤├───────┤                                      └───┘└───────┘»
q_4: ┤ H ├┤ Rz(0) ├────────────────────────────────────────────────────»
     ├───┤├───────┤                                                    »
q_5: ┤ H ├┤ Rz(0) ├────────────────────────────────────────────────────»
     ├───┤├───────┤                                                    »
q_6: ┤ H ├┤ Rz(0) ├────────────────────────────────